# NStream GPU Worker
Temporary authenticated MuseTalk 1.5 worker for testing NStream on a Colab T4 GPU. Do not use this ephemeral runtime as production infrastructure.

In [ ]:
# Upload gpu-deployment.zip using Colab's Files panel, then run this cell.
# This removes both possible old package paths and extracts without prompts.
import shutil, zipfile, pathlib
for path in (pathlib.Path('/content/nstream-gpu-deployment'), pathlib.Path('/content/gpu-deployment')):
    shutil.rmtree(path, ignore_errors=True)
with zipfile.ZipFile('/content/gpu-deployment.zip') as archive:
    archive.extractall('/content')
pathlib.Path('/content/gpu-deployment').rename('/content/nstream-gpu-deployment')
!chmod +x /content/nstream-gpu-deployment/setup_colab.sh /content/nstream-gpu-deployment/start_colab.sh /content/nstream-gpu-deployment/start_tunnel.sh
!bash /content/nstream-gpu-deployment/setup_colab.sh

## Start the worker
Set a new 32+ character token in the next cell. Colab keeps it only in this runtime's memory; do not place it in the notebook or share it. The worker cell starts the service in the background and verifies its local health endpoint.

In [ ]:
import getpass, os
token = getpass.getpass('Set a new 32+ character LIPSYNC_SERVICE_TOKEN: ')
if len(token) < 32:
    raise ValueError('Use a token of at least 32 characters.')
os.environ['LIPSYNC_SERVICE_TOKEN'] = token
del token


In [ ]:
%%bash
set -euo pipefail
export MPLBACKEND=Agg
nohup bash /content/nstream-gpu-deployment/start_colab.sh > /content/nstream-gpu-worker.log 2>&1 &
echo $! > /content/nstream-gpu-worker.pid
sleep 5
cat /content/nstream-gpu-worker.log
curl --fail --silent --show-error http://127.0.0.1:8000/health


## Create the temporary HTTPS tunnel
Run the next cell after health reports `ready: true`. Copy the `https://…trycloudflare.com` URL only; never share the bearer token.

In [ ]:
%%bash
set -euo pipefail
nohup bash /content/nstream-gpu-deployment/start_tunnel.sh > /content/nstream-gpu-tunnel.log 2>&1 &
for attempt in {1..20}; do
  if grep -Eo 'https://[-a-z0-9]+\.trycloudflare\.com' /content/nstream-gpu-tunnel.log; then exit 0; fi
  sleep 2
done
cat /content/nstream-gpu-tunnel.log
exit 1
